# Phase 7 Notebook 07: Dynamic Alpha Stress Testing v3

Purpose: stress-test constructed alpha candidates from the production 04A Dynamic Alpha Engine after 04B Alpha Walk-Forward Validation.

Core production path:
04A Dynamic Alpha Engine -> 04B Alpha Walk-Forward Validation -> 07 Alpha Stress Testing -> 08 Survivor Freeze -> 09 Portfolio Construction.

Scope boundaries:
- This notebook stress-tests constructed alpha candidates only.
- It does not use regime overlay diagnostics from Notebook 05 or 06.
- 05/06 overlays are diagnostic-only and remain excluded while `regime_overlay_diagnostic_decision_current` is `PARK_OVERLAYS`.
- It does not change signal formulas, alpha construction formulas, stress thresholds, survivor freeze logic, portfolio construction, or ML logic.


## 1. Imports and Config

In [1]:
from pathlib import Path
import gc
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.alpha_stress import (
    apply_alpha_stress_gate,
    build_alpha_panel,
    build_alpha_stress_audit_summary,
    build_alpha_stress_case_matrix,
    build_alpha_stress_degradation_matrix,
    load_constructed_alpha_stress_inputs,
    select_constructed_alpha_stress_candidates,
    stress_alpha_costs,
    stress_alpha_degradation,
    stress_alpha_execution_delay,
    stress_alpha_subperiods,
    stress_alpha_turnover,
    stress_alpha_universe_subsamples,
    summarize_alpha_stress_results,
)
from src.alpha_stress_storage import ALPHA_STRESS_TABLES, save_alpha_stress_outputs
from src.db import load_ohlcv_panels
from src.run_config import get_sqlite_db_path, make_run_id, make_run_timestamp

DB_PATH = get_sqlite_db_path()
ALPHA_STRESS_VERSION = "phase7_dynamic_alpha_stress_v3"

V3_DYNAMIC_ALPHAS = [
    "alpha_hybrid_adaptive_v3",
    "alpha_rolling_ic_dynamic_v3",
    "alpha_regime_blend_dynamic_v3",
    "alpha_decay_aware_dynamic_v3",
]

PRIORITY_ALPHAS = V3_DYNAMIC_ALPHAS + [
    "alpha_persistence_blend_v2",
    "alpha_diversified_research_v2",
    "alpha_health_weighted_research_v1",
    "alpha_equal_weight_research_v1",
    "alpha_smooth_regime_weighted_v2",
]

COST_BPS_LIST = [0, 5, 10, 25]
EXECUTION_DELAYS = [0, 1, 2, 5]
DEGRADATION_MULTIPLIERS = [0.75, 0.50]

pd.set_option("display.max_columns", 200)
DB_PATH


PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create stress run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase7_constructed_alpha_stress")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase7_constructed_alpha_stress_20260511_083012', '2026-05-11 08:30:12')

## 3. Load constructed alpha stress inputs

In [3]:
inputs = load_constructed_alpha_stress_inputs()

input_shapes = pd.DataFrame(
    [
        {"input_name": name, "n_rows": len(df), "n_columns": len(df.columns)}
        for name, df in inputs.items()
    ]
)
display(input_shapes)

,input_name,n_rows,n_columns
0,alpha_long,10028440,6
1,quality,10,13
2,diagnostics,10,16
3,wfv_gate,24,18
4,wfv_winner_summary,6,10
5,regime_overlay_decision,1,9


## 4. Select 04B constructed alpha stress candidates


In [4]:
stress_candidates = select_constructed_alpha_stress_candidates(
    alpha_quality=inputs["quality"],
    alpha_diagnostics=inputs["diagnostics"],
    constructed_alpha_wfv_gate=inputs["wfv_gate"],
    constructed_alpha_wfv_winner_summary=inputs["wfv_winner_summary"],
    priority_alphas=PRIORITY_ALPHAS,
)

regime_overlay_decision = inputs["regime_overlay_decision"].copy()
regime_overlay_current_decision = (
    regime_overlay_decision["diagnostic_decision"].dropna().astype(str).iloc[-1]
    if not regime_overlay_decision.empty and "diagnostic_decision" in regime_overlay_decision.columns
    else "UNKNOWN"
)
regime_overlay_exclusion_confirmation = pd.DataFrame(
    [
        {
            "diagnostic_path": "05/06 regime overlay diagnostics",
            "current_decision": regime_overlay_current_decision,
            "excluded_from_notebook_07": True,
            "notes": "Regime overlay candidates are diagnostic-only and are not stress-tested in the production alpha path.",
        }
    ]
)

v3_dynamic_alpha_inclusion_check = pd.DataFrame(
    [
        {
            "alpha_name": alpha_name,
            "included_in_stress_candidates": bool(stress_candidates["alpha_name"].astype(str).eq(alpha_name).any()),
        }
        for alpha_name in V3_DYNAMIC_ALPHAS
    ]
)

if stress_candidates.empty:
    print("No constructed alpha candidates met construction-quality and WFV-status filters.")
else:
    print(f"Constructed alpha stress candidates: {stress_candidates['alpha_name'].nunique()}")

print(f"Regime overlay diagnostic decision: {regime_overlay_current_decision}; overlays excluded from Notebook 07.")
display(regime_overlay_exclusion_confirmation)
display(v3_dynamic_alpha_inclusion_check)
display(stress_candidates)


Constructed alpha stress candidates: 3
Regime overlay diagnostic decision: PARK_OVERLAYS; overlays excluded from Notebook 07.


,diagnostic_path,current_decision,excluded_from_notebook_07,notes
0,05/06 regime overlay diagnostics,PARK_OVERLAYS,True,Regime overlay candidates are diagnostic-only ...


,alpha_name,included_in_stress_candidates
0,alpha_hybrid_adaptive_v3,False
1,alpha_rolling_ic_dynamic_v3,False
2,alpha_regime_blend_dynamic_v3,False
3,alpha_decay_aware_dynamic_v3,False


,alpha_name,construction_status,horizon,wfv_status,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,constructed_alpha_wfv_notes,run_id,constructed_alpha_wfv_version,avg_turnover_proxy,turnover_risk_flag,selection_priority
0,alpha_regime_blend_dynamic_v4_smooth,APPROVED_FOR_ALPHA_VALIDATION,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.046854,0.726320,0.5,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.746788,LOW_TURNOVER_RISK,9
1,alpha_orthogonal_diversifier_v2_score_weighted...,APPROVED_FOR_ALPHA_VALIDATION,20,APPROVED_CONSTRUCTED_ALPHA_WFV,0.037789,0.739775,1.0,0.75,Meets strict constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.946872,MODERATE_TURNOVER_RISK,9
2,alpha_hybrid_adaptive_v4_smooth,APPROVED_FOR_ALPHA_VALIDATION,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.034611,0.508163,0.5,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.738334,LOW_TURNOVER_RISK,9


## 5. Load constructed alpha panels

In [5]:
alpha_long = inputs["alpha_long"]
alpha_panels = {}

for _, candidate in stress_candidates.iterrows():
    alpha_name = candidate["alpha_name"]
    panel = build_alpha_panel(alpha_long, alpha_name)
    panel.attrs["alpha_name"] = alpha_name
    panel.attrs["avg_turnover_proxy"] = candidate.get("avg_turnover_proxy")
    panel.attrs["turnover_risk_flag"] = candidate.get("turnover_risk_flag")
    alpha_panels[alpha_name] = panel

panel_summary = pd.DataFrame(
    [
        {
            "alpha_name": alpha_name,
            "n_dates": panel.shape[0],
            "n_tickers": panel.shape[1],
            "finite_pct": float(panel.notna().to_numpy().mean()) if panel.size else 0.0,
            "avg_turnover_proxy": panel.attrs.get("avg_turnover_proxy"),
            "turnover_risk_flag": panel.attrs.get("turnover_risk_flag"),
        }
        for alpha_name, panel in alpha_panels.items()
    ]
)
display(panel_summary)

,alpha_name,n_dates,n_tickers,finite_pct,avg_turnover_proxy,turnover_risk_flag
0,alpha_regime_blend_dynamic_v4_smooth,2053,462,0.939456,1.746788,LOW_TURNOVER_RISK
1,alpha_orthogonal_diversifier_v2_score_weighted...,2053,462,0.938549,1.946872,MODERATE_TURNOVER_RISK
2,alpha_hybrid_adaptive_v4_smooth,2053,462,0.939219,1.738334,LOW_TURNOVER_RISK


## 6. Load clean close prices

In [6]:
ohlcv = load_ohlcv_panels(current=True, db_path=DB_PATH)
close_prices = ohlcv["close"]

close_prices.shape

(2098, 478)

## 7. Run constructed alpha stress tests

In [7]:
stress_frames = []

for _, candidate in stress_candidates.iterrows():
    alpha_name = candidate["alpha_name"]
    horizon = int(candidate["horizon"])
    panel = alpha_panels[alpha_name]

    stress_frames.extend(
        [
            stress_alpha_costs(panel, close_prices, horizon, cost_bps_list=COST_BPS_LIST),
            stress_alpha_execution_delay(panel, close_prices, horizon, delays=EXECUTION_DELAYS),
            stress_alpha_turnover(panel, close_prices, horizon),
            stress_alpha_subperiods(panel, close_prices, horizon),
            stress_alpha_universe_subsamples(panel, close_prices, horizon),
            stress_alpha_degradation(panel, close_prices, horizon, multipliers=DEGRADATION_MULTIPLIERS),
        ]
    )

alpha_stress_results = (
    pd.concat(stress_frames, ignore_index=True)
    if stress_frames
    else pd.DataFrame()
)

print(f"Stress result rows: {len(alpha_stress_results)}")
display(alpha_stress_results)

Stress result rows: 54


,alpha_name,horizon,stress_type,stress_case,effective_mean_ic,effective_ic_ir,degradation_from_base,pass_flag,avg_turnover_proxy,turnover_risk_flag,notes
0,alpha_regime_blend_dynamic_v4_smooth,20,cost,0bps,0.013391,0.102541,0.000000,True,1.746788,LOW_TURNOVER_RISK,turnover_proxy=1.7468
1,alpha_regime_blend_dynamic_v4_smooth,20,cost,5bps,0.012518,0.102541,0.065222,True,1.746788,LOW_TURNOVER_RISK,turnover_proxy=1.7468
2,alpha_regime_blend_dynamic_v4_smooth,20,cost,10bps,0.011644,0.102541,0.130444,True,1.746788,LOW_TURNOVER_RISK,turnover_proxy=1.7468
3,alpha_regime_blend_dynamic_v4_smooth,20,cost,25bps,0.009024,0.102541,0.326111,True,1.746788,LOW_TURNOVER_RISK,turnover_proxy=1.7468
4,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,0d_delay,0.013391,0.102541,0.000000,False,1.746788,LOW_TURNOVER_RISK,n_obs=569167
5,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,1d_delay,0.012789,0.098340,0.044955,True,1.746788,LOW_TURNOVER_RISK,n_obs=568863
6,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,2d_delay,0.012199,0.094120,0.089010,True,1.746788,LOW_TURNOVER_RISK,n_obs=568561
7,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,5d_delay,0.010605,0.082436,0.208086,True,1.746788,LOW_TURNOVER_RISK,n_obs=567651
8,alpha_regime_blend_dynamic_v4_smooth,20,turnover,LOW_TURNOVER_RISK,0.013391,0.102541,0.000000,True,1.746788,LOW_TURNOVER_RISK,avg_turnover_proxy=1.7468
9,alpha_regime_blend_dynamic_v4_smooth,20,subperiod,early,0.014086,0.108000,-0.051911,True,1.746788,LOW_TURNOVER_RISK,2018-03-08 to 2020-11-20; n_obs=183859


## 8. Build stress summary and gate

In [8]:
alpha_stress_summary = summarize_alpha_stress_results(alpha_stress_results)
alpha_stress_gate = apply_alpha_stress_gate(alpha_stress_summary)
alpha_stress_case_matrix = build_alpha_stress_case_matrix(alpha_stress_results)
alpha_stress_degradation_matrix = build_alpha_stress_degradation_matrix(alpha_stress_results)
alpha_stress_audit_summary = build_alpha_stress_audit_summary(alpha_stress_results, alpha_stress_gate)

stress_gate_counts = (
    alpha_stress_gate["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_alpha_horizons")
    if "status" in alpha_stress_gate.columns
    else pd.DataFrame(columns=["status", "n_alpha_horizons"])
)

survivor_tier_order = {
    "CORE_STRESS_SURVIVOR": 0,
    "BALANCED_STRESS_SURVIVOR": 1,
    "AGGRESSIVE_STRESS_SURVIVOR": 2,
    "WATCH_STRESS_SURVIVOR": 3,
    "NON_SURVIVOR": 4,
}
promotion_decision_order = {
    "PROMOTE_CORE": 0,
    "PROMOTE_BALANCED": 1,
    "REVIEW_SATELLITE": 2,
    "REJECT_HIGH_TURNOVER": 3,
    "REJECT": 4,
}

stress_summary_display = alpha_stress_summary.merge(
    alpha_stress_gate[
        [
            "alpha_name",
            "horizon",
            "status",
            "survivor_tier",
            "promotion_decision",
            "alpha_role",
            "failure_category",
            "interpretation_notes",
        ]
    ],
    on=["alpha_name", "horizon"],
    how="left",
)
stress_summary_display["survivor_tier_rank"] = stress_summary_display["survivor_tier"].map(survivor_tier_order).fillna(99)
stress_summary_display["promotion_decision_rank"] = stress_summary_display["promotion_decision"].map(promotion_decision_order).fillna(99)
stress_summary_display = stress_summary_display.sort_values(
    ["promotion_decision_rank", "pass_rate", "alpha_name"],
    ascending=[True, False, True],
).drop(columns=["survivor_tier_rank", "promotion_decision_rank"])

survivor_tier_counts = (
    alpha_stress_gate["survivor_tier"]
    .value_counts(dropna=False)
    .rename_axis("survivor_tier")
    .reset_index(name="n_alpha_horizons")
    if "survivor_tier" in alpha_stress_gate.columns
    else pd.DataFrame(columns=["survivor_tier", "n_alpha_horizons"])
)
promotion_decision_counts = (
    alpha_stress_gate["promotion_decision"]
    .value_counts(dropna=False)
    .rename_axis("promotion_decision")
    .reset_index(name="n_alpha_horizons")
    if "promotion_decision" in alpha_stress_gate.columns
    else pd.DataFrame(columns=["promotion_decision", "n_alpha_horizons"])
)
alpha_role_counts = (
    alpha_stress_gate["alpha_role"]
    .value_counts(dropna=False)
    .rename_axis("alpha_role")
    .reset_index(name="n_alpha_horizons")
    if "alpha_role" in alpha_stress_gate.columns
    else pd.DataFrame(columns=["alpha_role", "n_alpha_horizons"])
)

stress_gate_display = alpha_stress_gate.copy()
stress_gate_display["promotion_decision_rank"] = stress_gate_display["promotion_decision"].map(promotion_decision_order).fillna(99)
stress_gate_display = stress_gate_display.sort_values(
    ["promotion_decision_rank", "pass_rate", "alpha_name"],
    ascending=[True, False, True],
).drop(columns=["promotion_decision_rank"])

if "pass_flag" in alpha_stress_results.columns:
    failed_stress_cases = alpha_stress_results.loc[
        alpha_stress_results["pass_flag"].eq(False)
    ].sort_values(["alpha_name", "horizon", "stress_type", "stress_case"])
else:
    failed_stress_cases = pd.DataFrame(columns=["alpha_name", "horizon", "stress_type", "stress_case", "pass_flag"])

display(stress_gate_counts)
display(survivor_tier_counts)
display(promotion_decision_counts)
display(alpha_role_counts)
display(stress_summary_display)
display(stress_gate_display)
display(failed_stress_cases)


,status,n_alpha_horizons
0,APPROVED_STRESS,2
1,REJECTED_STRESS,1


,survivor_tier,n_alpha_horizons
0,WATCH_STRESS_SURVIVOR,2
1,NON_SURVIVOR,1


,promotion_decision,n_alpha_horizons
0,REVIEW_SATELLITE,2
1,REJECT,1


,alpha_role,n_alpha_horizons
0,SATELLITE_CANDIDATE,2
1,REJECTED_ALPHA,1


,alpha_name,horizon,n_stress_cases,n_passed,pass_rate,worst_degradation,catastrophic_degradation,cost_sensitivity_extreme,small_delay_failure,fragile_subset_warning,avg_turnover_proxy,turnover_risk_flag,high_turnover_weak_cost_robustness,failure_notes,status,survivor_tier,promotion_decision,alpha_role,failure_category,interpretation_notes
0,alpha_regime_blend_dynamic_v4_smooth,20,18,16,0.888889,0.422319,False,False,False,False,1.746788,LOW_TURNOVER_RISK,False,execution_delay/0d_delay: n_obs=569167; subper...,APPROVED_STRESS,WATCH_STRESS_SURVIVOR,REVIEW_SATELLITE,SATELLITE_CANDIDATE,NONE,High average performance but failed catastroph...
1,alpha_hybrid_adaptive_v4_smooth,20,18,15,0.833333,0.678978,False,False,False,False,1.738334,LOW_TURNOVER_RISK,False,cost/25bps: turnover_proxy=1.7383; execution_d...,APPROVED_STRESS,WATCH_STRESS_SURVIVOR,REVIEW_SATELLITE,SATELLITE_CANDIDATE,NONE,High average performance but failed catastroph...
2,alpha_orthogonal_diversifier_v2_score_weighted...,20,18,15,0.833333,0.897289,True,False,False,False,1.946872,MODERATE_TURNOVER_RISK,False,execution_delay/0d_delay: n_obs=569080; subper...,REJECTED_STRESS,NON_SURVIVOR,REJECT,REJECTED_ALPHA,MIXED_FAILURES,Rejected due to mixed stress failures.


,alpha_name,horizon,n_stress_cases,n_passed,pass_rate,worst_degradation,catastrophic_degradation,cost_sensitivity_extreme,small_delay_failure,fragile_subset_warning,avg_turnover_proxy,turnover_risk_flag,high_turnover_weak_cost_robustness,failure_notes,status,stress_gate_notes,survivor_tier,promotion_decision,alpha_role,failure_category,interpretation_notes
0,alpha_regime_blend_dynamic_v4_smooth,20,18,16,0.888889,0.422319,False,False,False,False,1.746788,LOW_TURNOVER_RISK,False,execution_delay/0d_delay: n_obs=569167; subper...,APPROVED_STRESS,Passes majority of stress cases with no fatal ...,WATCH_STRESS_SURVIVOR,REVIEW_SATELLITE,SATELLITE_CANDIDATE,NONE,High average performance but failed catastroph...
1,alpha_hybrid_adaptive_v4_smooth,20,18,15,0.833333,0.678978,False,False,False,False,1.738334,LOW_TURNOVER_RISK,False,cost/25bps: turnover_proxy=1.7383; execution_d...,APPROVED_STRESS,Passes majority of stress cases with no fatal ...,WATCH_STRESS_SURVIVOR,REVIEW_SATELLITE,SATELLITE_CANDIDATE,NONE,High average performance but failed catastroph...
2,alpha_orthogonal_diversifier_v2_score_weighted...,20,18,15,0.833333,0.897289,True,False,False,False,1.946872,MODERATE_TURNOVER_RISK,False,execution_delay/0d_delay: n_obs=569080; subper...,REJECTED_STRESS,catastrophic degradation,NON_SURVIVOR,REJECT,REJECTED_ALPHA,MIXED_FAILURES,Rejected due to mixed stress failures.


,alpha_name,horizon,stress_type,stress_case,effective_mean_ic,effective_ic_ir,degradation_from_base,pass_flag,avg_turnover_proxy,turnover_risk_flag,notes
39,alpha_hybrid_adaptive_v4_smooth,20,cost,25bps,0.007330,0.084797,0.372219,False,1.738334,LOW_TURNOVER_RISK,turnover_proxy=1.7383
40,alpha_hybrid_adaptive_v4_smooth,20,execution_delay,0d_delay,0.011675,0.084797,0.000000,False,1.738334,LOW_TURNOVER_RISK,n_obs=569098
46,alpha_hybrid_adaptive_v4_smooth,20,subperiod,middle,0.003748,0.026589,0.678978,False,1.738334,LOW_TURNOVER_RISK,2020-11-23 to 2023-08-14; n_obs=186638
22,alpha_orthogonal_diversifier_v2_score_weighted...,20,execution_delay,0d_delay,0.014127,0.104823,0.000000,False,1.946872,MODERATE_TURNOVER_RISK,n_obs=569080
27,alpha_orthogonal_diversifier_v2_score_weighted...,20,subperiod,early,0.001451,0.011220,0.897289,False,1.946872,MODERATE_TURNOVER_RISK,2018-03-08 to 2020-11-20; n_obs=183802
30,alpha_orthogonal_diversifier_v2_score_weighted...,20,universe_subsample,first_half_tickers,0.004953,0.033891,0.649375,False,1.946872,MODERATE_TURNOVER_RISK,n_tickers=231; n_obs=277319
4,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,0d_delay,0.013391,0.102541,0.000000,False,1.746788,LOW_TURNOVER_RISK,n_obs=569167
10,alpha_regime_blend_dynamic_v4_smooth,20,subperiod,middle,0.007736,0.056962,0.422319,False,1.746788,LOW_TURNOVER_RISK,2020-11-23 to 2023-08-14; n_obs=186663


## 9. Save outputs to SQLite

In [9]:
saved_paths = save_alpha_stress_outputs(
    stress_results=alpha_stress_results,
    stress_summary=alpha_stress_summary,
    stress_gate=alpha_stress_gate,
    stress_case_matrix=alpha_stress_case_matrix,
    stress_degradation_matrix=alpha_stress_degradation_matrix,
    stress_audit_summary=alpha_stress_audit_summary,
    db_path=DB_PATH,
    run_id=run_id,
    stress_version=ALPHA_STRESS_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in ALPHA_STRESS_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,results,alpha_stress_results_current,alpha_stress_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,alpha_stress_summary_current,alpha_stress_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,alpha_stress_gate_current,alpha_stress_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,case_matrix,alpha_stress_case_matrix_current,alpha_stress_case_matrix_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,degradation_matrix,alpha_stress_degradation_matrix_current,alpha_stress_degradation_matrix_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,audit_summary,alpha_stress_audit_summary_current,alpha_stress_audit_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 10. Final summary

In [10]:
final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "alpha_stress_version", "value": ALPHA_STRESS_VERSION},
        {"metric": "stress_candidate_count", "value": stress_candidates["alpha_name"].nunique()},
        {"metric": "stress_result_rows", "value": len(alpha_stress_results)},
        {"metric": "stress_gate_rows", "value": len(alpha_stress_gate)},
        {"metric": "regime_overlay_diagnostic_decision", "value": regime_overlay_current_decision},
        {"metric": "regime_overlay_candidates_excluded", "value": True},
    ]
)

print("Dynamic constructed alpha stress run summary")
display(final_summary)

print("Selected 04B stress candidates")
display(stress_candidates)

print("V3 dynamic alpha inclusion check")
display(v3_dynamic_alpha_inclusion_check)

print("Regime overlay exclusion / PARK_OVERLAYS confirmation")
display(regime_overlay_exclusion_confirmation)

print("Status counts")
display(stress_gate_counts)

print("Survivor tier counts")
display(survivor_tier_counts)

print("Promotion decision counts")
display(promotion_decision_counts)

print("Alpha role counts")
display(alpha_role_counts)

print("Stress gate sorted by promotion_decision and pass_rate")
display(stress_gate_display)

print("Stress summary sorted by pass_rate")
display(alpha_stress_summary.sort_values(["pass_rate", "alpha_name"], ascending=[False, True]))

print("Failed stress cases")
display(failed_stress_cases)

print("Degradation matrix")
display(alpha_stress_degradation_matrix)

print("SQLite tables written")
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving alpha stress outputs')


Dynamic constructed alpha stress run summary


,metric,value
0,run_id,phase7_constructed_alpha_stress_20260511_083012
1,run_timestamp,2026-05-11 08:30:12
2,alpha_stress_version,phase7_dynamic_alpha_stress_v3
3,stress_candidate_count,3
4,stress_result_rows,54
5,stress_gate_rows,3
6,regime_overlay_diagnostic_decision,PARK_OVERLAYS
7,regime_overlay_candidates_excluded,True


Selected 04B stress candidates


,alpha_name,construction_status,horizon,wfv_status,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,constructed_alpha_wfv_notes,run_id,constructed_alpha_wfv_version,avg_turnover_proxy,turnover_risk_flag,selection_priority
0,alpha_regime_blend_dynamic_v4_smooth,APPROVED_FOR_ALPHA_VALIDATION,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.046854,0.726320,0.5,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.746788,LOW_TURNOVER_RISK,9
1,alpha_orthogonal_diversifier_v2_score_weighted...,APPROVED_FOR_ALPHA_VALIDATION,20,APPROVED_CONSTRUCTED_ALPHA_WFV,0.037789,0.739775,1.0,0.75,Meets strict constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.946872,MODERATE_TURNOVER_RISK,9
2,alpha_hybrid_adaptive_v4_smooth,APPROVED_FOR_ALPHA_VALIDATION,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.034611,0.508163,0.5,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1,1.738334,LOW_TURNOVER_RISK,9


V3 dynamic alpha inclusion check


...

,alpha_name,horizon,stress_type,stress_case,effective_mean_ic,effective_ic_ir,degradation_from_base,pass_flag,avg_turnover_proxy,turnover_risk_flag,notes
39,alpha_hybrid_adaptive_v4_smooth,20,cost,25bps,0.007330,0.084797,0.372219,False,1.738334,LOW_TURNOVER_RISK,turnover_proxy=1.7383
40,alpha_hybrid_adaptive_v4_smooth,20,execution_delay,0d_delay,0.011675,0.084797,0.000000,False,1.738334,LOW_TURNOVER_RISK,n_obs=569098
46,alpha_hybrid_adaptive_v4_smooth,20,subperiod,middle,0.003748,0.026589,0.678978,False,1.738334,LOW_TURNOVER_RISK,2020-11-23 to 2023-08-14; n_obs=186638
22,alpha_orthogonal_diversifier_v2_score_weighted...,20,execution_delay,0d_delay,0.014127,0.104823,0.000000,False,1.946872,MODERATE_TURNOVER_RISK,n_obs=569080
27,alpha_orthogonal_diversifier_v2_score_weighted...,20,subperiod,early,0.001451,0.011220,0.897289,False,1.946872,MODERATE_TURNOVER_RISK,2018-03-08 to 2020-11-20; n_obs=183802
30,alpha_orthogonal_diversifier_v2_score_weighted...,20,universe_subsample,first_half_tickers,0.004953,0.033891,0.649375,False,1.946872,MODERATE_TURNOVER_RISK,n_tickers=231; n_obs=277319
4,alpha_regime_blend_dynamic_v4_smooth,20,execution_delay,0d_delay,0.013391,0.102541,0.000000,False,1.746788,LOW_TURNOVER_RISK,n_obs=569167
10,alpha_regime_blend_dynamic_v4_smooth,20,subperiod,middle,0.007736,0.056962,0.422319,False,1.746788,LOW_TURNOVER_RISK,2020-11-23 to 2023-08-14; n_obs=186663


Degradation matrix


,alpha_name,horizon,alpha_degradation__alpha_x_0.50,alpha_degradation__alpha_x_0.75,cost__0bps,cost__10bps,cost__25bps,cost__5bps,execution_delay__0d_delay,execution_delay__1d_delay,execution_delay__2d_delay,execution_delay__5d_delay,subperiod__early,subperiod__middle,subperiod__recent,turnover__LOW_TURNOVER_RISK,turnover__MODERATE_TURNOVER_RISK,universe_subsample__first_half_tickers,universe_subsample__random_half_seed_42,universe_subsample__random_half_seed_99,universe_subsample__second_half_tickers
0,alpha_hybrid_adaptive_v4_smooth,20,0.0,0.0,0.0,0.148888,0.372219,0.074444,0.0,0.049441,0.098489,0.235889,-0.029510,0.678978,-1.513359,0.0,NaN,-0.031260,0.089532,-0.107321,0.052427
1,alpha_orthogonal_diversifier_v2_score_weighted...,20,0.0,0.0,0.0,0.137815,0.344538,0.068908,0.0,0.037900,0.072862,0.169625,0.897289,-0.031512,-1.052437,NaN,0.0,0.649375,-0.097711,-0.443129,-0.598083
2,alpha_regime_blend_dynamic_v4_smooth,20,0.0,0.0,0.0,0.130444,0.326111,0.065222,0.0,0.044955,0.089010,0.208086,-0.051911,0.422319,-0.970098,0.0,NaN,-0.002760,0.122140,-0.064497,0.019862


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,results,alpha_stress_results_current,alpha_stress_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,alpha_stress_summary_current,alpha_stress_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,alpha_stress_gate_current,alpha_stress_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,case_matrix,alpha_stress_case_matrix_current,alpha_stress_case_matrix_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,degradation_matrix,alpha_stress_degradation_matrix_current,alpha_stress_degradation_matrix_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,audit_summary,alpha_stress_audit_summary_current,alpha_stress_audit_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
